# Transaction Linking Evaluation

Scores transaction linking model results against ground truth — **no model inference required**.

**Inputs:**
- `evaluation_data/transaction_link_ground_truth.csv` — human-annotated ground truth
- `evaluation_data/transaction_link_model_results.csv` — model extraction output

**Fields evaluated (F1 scored):**
- `RECEIPT_DATE`, `RECEIPT_DESCRIPTION`, `RECEIPT_TOTAL`
- `BANK_TRANSACTION_DATE`, `BANK_TRANSACTION_DESCRIPTION`, `BANK_TRANSACTION_DEBIT`

**Metadata fields (captured, not scored):**
- `MATCHED_TRANSACTION` — match outcome (FOUND/PARTIAL_MATCH/NOT_FOUND)
- `CONFIDENCE` — model self-assessed match confidence (HIGH/MEDIUM/LOW)
- `MISMATCH_TYPE` — structured mismatch category
- `REASONING` — model explanation of match or mismatch decision

**Ground truth annotation fields (scored categorically):**
- `EXPECTED_MATCH_STATUS` — human-annotated expected outcome
- `EXPECTED_MISMATCH_TYPE` — human-annotated mismatch category
- `REASONING_QUALITY` — human-annotated reasoning usefulness (post-run annotation)
- `ANNOTATOR_NOTES` — freetext annotator explanation for ground truth decisions

Both CSVs share the same format: one row per receipt image, pipe-delimited for multi-receipt pages.

In [1]:
"""Cell 1: Imports and setup."""

import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.batch_analytics import BatchAnalytics  # noqa: E402
from common.batch_reporting import BatchReporter  # noqa: E402
from common.batch_visualizations import BatchVisualizer  # noqa: E402
from common.evaluation_metrics import (  # noqa: E402
    calculate_field_accuracy_with_method,
    load_ground_truth,
)
from common.field_config import get_document_type_fields  # noqa: E402

print("Imports loaded successfully.")

Imports loaded successfully.


In [2]:
"""Cell 2: Load ground truth and model results."""

GROUND_TRUTH_PATH = "evaluation_data/transaction_link_ground_truth.csv"
MODEL_RESULTS_PATH = "evaluation_data/transaction_link_model_results.csv"

ground_truth = load_ground_truth(GROUND_TRUTH_PATH, show_sample=True)
model_results = load_ground_truth(MODEL_RESULTS_PATH, verbose=False)

gt_images = set(ground_truth.keys())
mr_images = set(model_results.keys())

missing_in_gt = mr_images - gt_images
extra_in_gt = gt_images - mr_images

if missing_in_gt:
    print(
        f"Warning: {len(missing_in_gt)} images in model results not found in "
        f"ground truth: {sorted(missing_in_gt)}"
    )
if extra_in_gt:
    print(
        f"Info: {len(extra_in_gt)} ground truth images not in model results "
        f"(will be skipped): {sorted(extra_in_gt)}"
    )

common_images = gt_images & mr_images
print(f"\n{len(common_images)} images to evaluate")

📊 Ground truth CSV loaded with 3 rows and 9 columns
📋 Available columns: ['image_file', 'bank_statement_file', 'DOCUMENT_TYPE', 'RECEIPT_DATE', 'RECEIPT_DESCRIPTION', 'RECEIPT_TOTAL', 'BANK_TRANSACTION_DATE', 'BANK_TRANSACTION_DESCRIPTION', 'BANK_TRANSACTION_DEBIT']
✅ Using 'image_file' as image identifier column
📄 Sample ground truth data:
                           image_file          bank_statement_file    DOCUMENT_TYPE                         RECEIPT_DATE                                                    RECEIPT_DESCRIPTION             RECEIPT_TOTAL                BANK_TRANSACTION_DATE                                 BANK_TRANSACTION_DESCRIPTION    BANK_TRANSACTION_DEBIT
     synthetic_multi_receipt_page.png synthetic_bank_statement.png TRANSACTION_LINK 15/01/2024 | 12/01/2024 | 10/01/2024 Office Supplies Plus | MetroCafe & Grill | Sydney Auto Parts Warehouse $83.48 | $39.70 | $142.80 15/01/2024 | 12/01/2024 | 10/01/2024 OFFICE SUPPLIES PLU | METROCAFE ANDGRI | SYDNEY AUTO PARTS W

In [ ]:
"""Cell 3: Determine evaluation fields for transaction_link."""

# Fields from field_definitions.yaml for transaction_link
all_fields = get_document_type_fields("transaction_link")
print(f"All transaction_link fields ({len(all_fields)}): {all_fields}")

# Exclude metadata columns that aren't model outputs to score
METADATA_FIELDS = {
    "DOCUMENT_TYPE",
    "BANK_STATEMENT_FILE",
    "MATCHED_TRANSACTION",
    "CONFIDENCE",
    "MISMATCH_TYPE",
    "REASONING",
}
eval_fields = [f for f in all_fields if f not in METADATA_FIELDS]
print(f"\nEvaluation fields ({len(eval_fields)}): {eval_fields}")
print(f"Metadata fields (captured, not scored): {sorted(METADATA_FIELDS)}")

In [4]:
"""Cell 4: Build batch_results from model results CSV."""

batch_results = []
processing_times = []

for image_name in sorted(common_images):
    extracted_data = {
        k: v for k, v in model_results[image_name].items() if k != "image_file"
    }

    batch_results.append(
        {
            "image_name": image_name,
            "document_type": "transaction_link",
            "processing_time": 0.0,
            "prompt_used": "staged_transaction_linking",
            "extracted_data": extracted_data,
        }
    )
    processing_times.append(0.0)

print(f"Built {len(batch_results)} results from model CSV.")

Built 3 results from model CSV.


In [5]:
"""Cell 5: Score all fields for all images."""

for result in batch_results:
    image_name = result["image_name"]
    extracted = result["extracted_data"]
    gt = ground_truth.get(image_name, {})

    field_accuracies = {}
    for field in eval_fields:
        ext_val = extracted.get(field, "NOT_FOUND")
        gt_val = gt.get(field, "NOT_FOUND")
        metrics = calculate_field_accuracy_with_method(
            ext_val, gt_val, field, method="order_aware_f1"
        )
        field_accuracies[field] = {"accuracy": metrics["f1_score"]}

    overall = (
        sum(d["accuracy"] for d in field_accuracies.values()) / len(field_accuracies)
        if field_accuracies
        else 0.0
    )
    matched = sum(1 for d in field_accuracies.values() if d["accuracy"] >= 0.5)

    result["evaluation"] = {
        "overall_accuracy": overall,
        "fields_extracted": len(field_accuracies),
        "fields_matched": matched,
        "total_fields": len(eval_fields),
        "field_accuracies": field_accuracies,
    }

    print(f"{image_name}: {overall:.1%} ({matched}/{len(eval_fields)} fields matched)")
    # Show per-field detail
    for field, acc in field_accuracies.items():
        score = acc["accuracy"]
        status = "PASS" if score >= 1.0 else "PARTIAL" if score >= 0.5 else "FAIL"
        if score < 1.0:
            print(f"  [{status}] {field}: {score:.1%}")
            print(f"    got:      {extracted.get(field, 'NOT_FOUND')}")
            print(f"    expected: {gt.get(field, 'NOT_FOUND')}")

avg_acc = sum(r["evaluation"]["overall_accuracy"] for r in batch_results) / len(
    batch_results
)
print(f"\nBatch Average: {avg_acc:.1%}")

synthetic_dual_receipt_jbhifi_dymocks.png: 94.4% (6/6 fields matched)
  [PARTIAL] BANK_TRANSACTION_DESCRIPTION: 66.7%
    got:      ELECTRONICS STORE JB HI-FI AU | BOOKSTORE MELBOURNE CBD AU AUS
    expected: ELECTRONICS STORE JB HIFI AU AUS | BOOKSTORE MELBOURNE CBD AU AUS
synthetic_multi_receipt_page.png: 100.0% (6/6 fields matched)
synthetic_single_receipt_bunnings.png: 100.0% (6/6 fields matched)

Batch Average: 98.1%


In [ ]:
"""Cell 5b: Display match status, confidence, mismatch type, and reasoning per image."""

metadata_fields = ["MATCHED_TRANSACTION", "CONFIDENCE", "MISMATCH_TYPE", "REASONING"]
has_metadata = any(
    result["extracted_data"].get(f) for result in batch_results for f in metadata_fields
)

if not has_metadata:
    print("No metadata fields found in model results CSV.")
    print("Re-run staged_transaction_linking.ipynb to populate these fields.")
else:
    import pandas as pd  # noqa: E402

    pd.set_option("display.max_colwidth", None)

    def _split_pipe(extracted, field):
        val = extracted.get(field, "")
        return [v.strip() for v in val.split("|")] if val else []

    rows = []
    for result in batch_results:
        image_name = result["image_name"]
        extracted = result["extracted_data"]

        statuses = _split_pipe(extracted, "MATCHED_TRANSACTION")
        confidences = _split_pipe(extracted, "CONFIDENCE")
        mismatch_types = _split_pipe(extracted, "MISMATCH_TYPE")
        reasonings = _split_pipe(extracted, "REASONING")

        n = max(len(statuses), len(confidences), len(reasonings), 1)
        for i in range(n):
            rows.append(
                {
                    "Image": image_name if i == 0 else "",
                    "Receipt": i + 1,
                    "Status": statuses[i] if i < len(statuses) else "",
                    "Confidence": confidences[i] if i < len(confidences) else "",
                    "Mismatch Type": mismatch_types[i]
                    if i < len(mismatch_types)
                    else "",
                    "Reasoning": reasonings[i] if i < len(reasonings) else "",
                }
            )

    df_metadata = pd.DataFrame(rows)
    print("=== Match Status, Confidence & Reasoning ===")
    display(df_metadata)

    pd.reset_option("display.max_colwidth")

    # Match status distribution
    all_statuses = []
    for result in batch_results:
        val = result["extracted_data"].get("MATCHED_TRANSACTION", "")
        if val:
            all_statuses.extend(v.strip().upper() for v in val.split("|"))

    if all_statuses:
        status_counts = pd.Series(all_statuses).value_counts()
        print(f"\nMatch status distribution ({len(all_statuses)} receipts):")
        for level in ["FOUND", "PARTIAL_MATCH", "NOT_FOUND"]:
            count = status_counts.get(level, 0)
            pct = count / len(all_statuses) * 100
            print(f"  {level}: {count} ({pct:.0f}%)")

    # Confidence distribution
    all_confidences = []
    for result in batch_results:
        conf = result["extracted_data"].get("CONFIDENCE", "")
        if conf:
            all_confidences.extend(c.strip().upper() for c in conf.split("|"))

    if all_confidences:
        conf_counts = pd.Series(all_confidences).value_counts()
        print(f"\nConfidence distribution ({len(all_confidences)} receipts):")
        for level in ["HIGH", "MEDIUM", "LOW"]:
            count = conf_counts.get(level, 0)
            pct = count / len(all_confidences) * 100
            print(f"  {level}: {count} ({pct:.0f}%)")

    # Mismatch type distribution (only for non-FOUND)
    all_mismatch = []
    for result in batch_results:
        val = result["extracted_data"].get("MISMATCH_TYPE", "")
        if val:
            all_mismatch.extend(v.strip().upper() for v in val.split("|"))

    non_none = [m for m in all_mismatch if m not in ("NONE", "NOT_FOUND", "")]
    if non_none:
        mismatch_counts = pd.Series(non_none).value_counts()
        print(f"\nMismatch type distribution ({len(non_none)} mismatched receipts):")
        for mtype, count in mismatch_counts.items():
            pct = count / len(non_none) * 100
            print(f"  {mtype}: {count} ({pct:.0f}%)")

In [ ]:
"""Cell 5c: Score match status and mismatch type against ground truth."""

import pandas as pd  # noqa: E402

# Check if ground truth has the annotation columns
gt_has_status = any("EXPECTED_MATCH_STATUS" in gt for gt in ground_truth.values())
mr_has_status = any(
    result["extracted_data"].get("MATCHED_TRANSACTION") for result in batch_results
)

if not gt_has_status:
    print("Ground truth CSV missing EXPECTED_MATCH_STATUS column.")
    print("Add this column to enable match status scoring.")
elif not mr_has_status:
    print("Model results CSV missing MATCHED_TRANSACTION column.")
    print("Re-run staged_transaction_linking.ipynb with updated prompts.")
else:
    pd.set_option("display.max_colwidth", None)

    def _split_pipe_field(data, field):
        val = data.get(field, "")
        return [v.strip() for v in val.split("|")] if val else []

    # --- Match status accuracy ---
    status_rows = []
    status_correct = 0
    status_total = 0

    for result in batch_results:
        image_name = result["image_name"]
        extracted = result["extracted_data"]
        gt = ground_truth.get(image_name, {})

        pred_statuses = _split_pipe_field(extracted, "MATCHED_TRANSACTION")
        gt_statuses = _split_pipe_field(gt, "EXPECTED_MATCH_STATUS")
        annotator_notes = gt.get("ANNOTATOR_NOTES", "")

        n = max(len(pred_statuses), len(gt_statuses))
        for i in range(n):
            pred = pred_statuses[i].upper() if i < len(pred_statuses) else "MISSING"
            expected = gt_statuses[i].upper() if i < len(gt_statuses) else "MISSING"
            match = pred == expected
            status_correct += int(match)
            status_total += 1
            status_rows.append(
                {
                    "Image": image_name if i == 0 else "",
                    "Receipt": i + 1,
                    "Predicted Status": pred,
                    "Expected Status": expected,
                    "Correct": "PASS" if match else "FAIL",
                    "Annotator Notes": annotator_notes if i == 0 else "",
                }
            )

    df_status = pd.DataFrame(status_rows)
    print("=== Match Status Accuracy ===")
    display(df_status)
    if status_total > 0:
        print(
            f"\nMatch status accuracy: {status_correct}/{status_total} "
            f"({status_correct / status_total:.0%})"
        )

    # --- Mismatch type accuracy (for PARTIAL_MATCH and NOT_FOUND only) ---
    gt_has_mismatch = any(
        "EXPECTED_MISMATCH_TYPE" in gt for gt in ground_truth.values()
    )
    mr_has_mismatch = any(
        result["extracted_data"].get("MISMATCH_TYPE") for result in batch_results
    )

    if gt_has_mismatch and mr_has_mismatch:
        mismatch_rows = []
        mismatch_correct = 0
        mismatch_total = 0

        for result in batch_results:
            image_name = result["image_name"]
            extracted = result["extracted_data"]
            gt = ground_truth.get(image_name, {})

            pred_statuses = _split_pipe_field(extracted, "MATCHED_TRANSACTION")
            pred_mismatches = _split_pipe_field(extracted, "MISMATCH_TYPE")
            gt_mismatches = _split_pipe_field(gt, "EXPECTED_MISMATCH_TYPE")

            n = max(len(pred_mismatches), len(gt_mismatches))
            for i in range(n):
                status = (
                    pred_statuses[i].upper() if i < len(pred_statuses) else "MISSING"
                )
                # Only score mismatch type for non-FOUND results
                if status in ("PARTIAL_MATCH", "NOT_FOUND"):
                    pred = (
                        pred_mismatches[i].upper()
                        if i < len(pred_mismatches)
                        else "MISSING"
                    )
                    expected = (
                        gt_mismatches[i].upper()
                        if i < len(gt_mismatches)
                        else "MISSING"
                    )
                    match = pred == expected
                    mismatch_correct += int(match)
                    mismatch_total += 1
                    mismatch_rows.append(
                        {
                            "Image": image_name if i == 0 else "",
                            "Receipt": i + 1,
                            "Status": status,
                            "Predicted Mismatch": pred,
                            "Expected Mismatch": expected,
                            "Correct": "PASS" if match else "FAIL",
                        }
                    )

        if mismatch_rows:
            df_mismatch = pd.DataFrame(mismatch_rows)
            print("\n=== Mismatch Type Accuracy (PARTIAL_MATCH/NOT_FOUND only) ===")
            display(df_mismatch)
            print(
                f"\nMismatch type accuracy: {mismatch_correct}/{mismatch_total} "
                f"({mismatch_correct / mismatch_total:.0%})"
            )
        else:
            print("\nNo PARTIAL_MATCH or NOT_FOUND results to score mismatch types.")
    elif not gt_has_mismatch:
        print(
            "\nGround truth CSV missing EXPECTED_MISMATCH_TYPE — skipping mismatch scoring."
        )
    else:
        print("\nModel results CSV missing MISMATCH_TYPE — skipping mismatch scoring.")

    pd.reset_option("display.max_colwidth")

In [ ]:
"""Cell 5d: Confidence calibration — accuracy by confidence level."""

import pandas as pd  # noqa: E402

gt_has_status = any("EXPECTED_MATCH_STATUS" in gt for gt in ground_truth.values())
mr_has_confidence = any(
    result["extracted_data"].get("CONFIDENCE") for result in batch_results
)
mr_has_status = any(
    result["extracted_data"].get("MATCHED_TRANSACTION") for result in batch_results
)

if not (gt_has_status and mr_has_confidence and mr_has_status):
    print(
        "Confidence calibration requires MATCHED_TRANSACTION, CONFIDENCE in model results"
    )
    print("and EXPECTED_MATCH_STATUS in ground truth. Skipping.")
else:

    def _split_pipe_cal(data, field):
        val = data.get(field, "")
        return [v.strip().upper() for v in val.split("|")] if val else []

    calibration_rows = []
    for result in batch_results:
        image_name = result["image_name"]
        extracted = result["extracted_data"]
        gt = ground_truth.get(image_name, {})

        pred_statuses = _split_pipe_cal(extracted, "MATCHED_TRANSACTION")
        gt_statuses = _split_pipe_cal(gt, "EXPECTED_MATCH_STATUS")
        confidences = _split_pipe_cal(extracted, "CONFIDENCE")

        n = min(len(pred_statuses), len(gt_statuses), len(confidences))
        for i in range(n):
            calibration_rows.append(
                {
                    "confidence": confidences[i],
                    "status_correct": pred_statuses[i] == gt_statuses[i],
                }
            )

    if calibration_rows:
        df_cal = pd.DataFrame(calibration_rows)
        print("=== Confidence Calibration ===")
        print("(Are HIGH-confidence matches more accurate than LOW?)\n")

        for level in ["HIGH", "MEDIUM", "LOW"]:
            subset = df_cal[df_cal["confidence"] == level]
            if len(subset) > 0:
                acc = subset["status_correct"].mean()
                print(
                    f"  {level}: {subset['status_correct'].sum()}/{len(subset)} "
                    f"correct ({acc:.0%})"
                )
            else:
                print(f"  {level}: no receipts at this level")

        total_acc = df_cal["status_correct"].mean()
        print(
            f"\n  Overall: {df_cal['status_correct'].sum()}/{len(df_cal)} "
            f"correct ({total_acc:.0%})"
        )
    else:
        print("No calibration data available — re-run inference with updated prompts.")

In [ ]:
"""Cell 5e: Reasoning quality distribution (requires human annotation)."""

import pandas as pd  # noqa: E402

gt_has_quality = any("REASONING_QUALITY" in gt for gt in ground_truth.values())

if not gt_has_quality:
    print("Ground truth CSV missing REASONING_QUALITY column.")
    print("To enable: after a model run, add REASONING_QUALITY to ground truth CSV")
    print(
        "with values: USEFUL | PARTIALLY_USEFUL | NOT_USEFUL (pipe-delimited per receipt)."
    )
else:

    def _split_pipe_rq(data, field):
        val = data.get(field, "")
        return [v.strip().upper() for v in val.split("|")] if val else []

    all_qualities = []
    for result in batch_results:
        image_name = result["image_name"]
        gt = ground_truth.get(image_name, {})
        qualities = _split_pipe_rq(gt, "REASONING_QUALITY")
        all_qualities.extend(qualities)

    if all_qualities:
        quality_counts = pd.Series(all_qualities).value_counts()
        print(
            f"=== Reasoning Quality Distribution ({len(all_qualities)} receipts) ===\n"
        )
        for level in ["USEFUL", "PARTIALLY_USEFUL", "NOT_USEFUL"]:
            count = quality_counts.get(level, 0)
            pct = count / len(all_qualities) * 100
            bar = "#" * int(pct / 2)
            print(f"  {level:20s} {count:3d} ({pct:4.0f}%) {bar}")
    else:
        print("REASONING_QUALITY column exists but has no data.")

In [6]:
"""Cell 6: Analytics — DataFrames and summary statistics."""

analytics = BatchAnalytics(batch_results, processing_times)

df_results = analytics.create_results_dataframe()
df_summary = analytics.create_summary_statistics(df_results)
df_fields = analytics.create_field_statistics(df_results)

print("=== Summary Statistics ===")
display(df_summary)

if df_fields is not None:
    print("\n=== Field-Level Statistics ===")
    display(df_fields)

=== Summary Statistics ===


/home/jovyan/nfs_share/tod_2026/LMM_POC/common/batch_analytics.py:122: RuntimeWarning: divide by zero encountered in scalar divide
  "Throughput (images/min)": 60 / np.mean(self.processing_times)


,Value
Total Images,3.000000
Successful Extractions,3.000000
Failed Extractions,0.000000
Average Accuracy (%),98.148148
Median Accuracy (%),100.000000
Min Accuracy (%),94.444444
Max Accuracy (%),100.000000
Average Processing Time (s),0.000000
Total Processing Time (s),0.000000
Throughput (images/min),inf



=== Field-Level Statistics ===


,Average Accuracy (%),Min Accuracy (%),Max Accuracy (%),Std Dev (%)
RECEIPT_DATE,100.00,100.00,100.0,0.00
RECEIPT_DESCRIPTION,100.00,100.00,100.0,0.00
RECEIPT_TOTAL,100.00,100.00,100.0,0.00
BANK_TRANSACTION_DATE,100.00,100.00,100.0,0.00
BANK_TRANSACTION_DEBIT,100.00,100.00,100.0,0.00
BANK_TRANSACTION_DESCRIPTION,88.89,66.67,100.0,19.25


In [7]:
"""Cell 7: Reporting — Markdown executive summary."""

doc_types_found = dict(Counter(r["document_type"] for r in batch_results))
df_doctype = analytics.create_doctype_statistics(df_results)
timestamp = "transaction_linking_eval"

reporter = BatchReporter(batch_results, processing_times, doc_types_found, timestamp)
md_report = reporter.generate_executive_summary(df_results, df_doctype, Path("output"))
print(md_report)

# Vision Model Batch Processing Report

**Generated:** 2026-03-17 23:23:40
**Batch ID:** transaction_linking_eval
**Model:** Unknown Version

## Executive Summary

### Overall Performance
- **Total Images Processed:** 3
- **Successful Extractions:** 3 (100.0%)
- **Average Accuracy:** 98.15%
- **Status:** ✅ **Production Ready**

### Processing Efficiency
- **Total Processing Time:** 0.00 seconds (0.0 minutes)
- **Average Time per Image:** 0.00 seconds
- **Throughput:** inf images/minute

### Document Type Distribution
- **transaction_link:** 3 (100.0%)

### Accuracy by Document Type
- **transaction_link:** 98.15%

### Top Performing Images
- synthetic_multi_receipt_page.png: 100.0% (transaction_link)
- synthetic_single_receipt_bunnings.png: 100.0% (transaction_link)
- synthetic_dual_receipt_jbhifi_dymocks.png: 94.4% (transaction_link)

## Output Files Generated

All results have been saved to: `output`

- **CSV Files:** `csv/batch_transaction_linking_eval_*.csv`
- **Visualizations:** `v

/home/jovyan/nfs_share/tod_2026/LMM_POC/common/batch_reporting.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  throughput = 60 / np.mean(self.processing_times) if self.processing_times else 0


In [ ]:
"""Cell 8: Visualizations — dashboard charts."""

visualizer = BatchVisualizer()

visualizer.create_dashboard(
    df_results,
    df_doctype,
    timestamp,
    save_path=None,
    show=True,
)

visualizer.create_field_heatmap(
    df_results,
    timestamp,
    save_path=None,
    show=True,
    field_order=eval_fields,
)